# 01A · IBD (GSE235236) — Intake, QC, and DEGs (Beginner-Friendly)

**Owner:** Sai

## Goal
From the IBD TPM matrix, produce:
- `data_processed/ibd/ibd_log2_tpm.csv`
- `data_processed/ibd/ibd_metadata_clean.csv`
- DEGs:
  - `results/de/ibd/IBD_DEG_UC_vs_Control.csv`
  - `results/de/ibd/IBD_DEG_CD_vs_Control.csv`

## Files required in Drive
Put these inside:
`{BASE}/data_raw/ibd/`

- `GSE235236_TPM_matrix_with_symbols.csv`
- `ibd_metadata.csv`  (must contain columns: `sample_id`, `group`)

If DEG fails: usually sample IDs in metadata do not match TPM column names.




### Mount + paths


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/GeneticCodonShared/Hippo_Dysbiosis" # change this to your own path

RAW_TPM  = f"{BASE}/data_raw/ibd/GSE235236_TPM_matrix_with_symbols.csv"
RAW_META = f"{BASE}/data_raw/ibd/ibd_metadata.csv"

OUT_PROC = f"{BASE}/data_processed/ibd"
OUT_DE   = f"{BASE}/results/de/ibd"
OUT_FIG  = f"{BASE}/results/figures/ibd"

import os
os.makedirs(OUT_PROC, exist_ok=True)
os.makedirs(OUT_DE,   exist_ok=True)
os.makedirs(OUT_FIG,  exist_ok=True)

print("TPM:", RAW_TPM)
print("META:", RAW_META)

### Import libraries/packages


In [ ]:
import pandas as pd, numpy as np
from scipy.stats import ttest_ind
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt


### Load TPM matrix


In [ ]:
tpm = pd.read_csv(RAW_TPM)
print("Raw TPM shape:", tpm.shape)
print("First columns:", list(tpm.columns[:6]))

# detect gene column
gene_col = None
for c in tpm.columns[:5]:
    if str(c).lower() in ["gene","symbol","hugo_symbol","gene_symbol","genesymbol"]:
        gene_col = c
        break
if gene_col is None:
    gene_col = tpm.columns[0]
print("Using gene column:", gene_col)

tpm = tpm.dropna(subset=[gene_col])
tpm[gene_col] = tpm[gene_col].astype(str).str.strip()
tpm = tpm.set_index(gene_col)

tpm = tpm.apply(pd.to_numeric, errors="coerce")
display(tpm.head())

### QC + cleaning


In [ ]:
print("Total NA:", int(tpm.isna().sum().sum()))
tpm = tpm.fillna(0)

dup = int(tpm.index.duplicated().sum())
print("Duplicate genes:", dup)
if dup:
    tpm = tpm.groupby(tpm.index).mean()

print("Clean TPM shape:", tpm.shape)

# outlier check
sums = tpm.sum(axis=0)
plt.figure(figsize=(6,4))
plt.hist(sums, bins=30)
plt.title("IBD: Sum(TPM) per sample (outlier check)")
plt.xlabel("Sum TPM"); plt.ylabel("Samples")
plt.tight_layout(); plt.show()

tpm.to_csv(f"{OUT_PROC}/ibd_tpm_clean.csv")
print("Saved ->", f"{OUT_PROC}/ibd_tpm_clean.csv")

### Load metadata + align samples


In [ ]:
meta = pd.read_csv(RAW_META)
display(meta.head())

need = {"sample_id","group"}
missing = need - set(meta.columns)
if missing:
    raise ValueError(f"Metadata missing columns: {missing}")

meta["sample_id"] = meta["sample_id"].astype(str).str.strip()
meta["group"]     = meta["group"].astype(str).str.strip()

tpm_cols  = pd.Index(tpm.columns.astype(str).str.strip())
meta_ids  = pd.Index(meta["sample_id"].astype(str).str.strip())
overlap   = tpm_cols.intersection(meta_ids)

print(f"Overlap samples: {len(overlap)} / TPM={tpm.shape[1]} / META={meta.shape[0]}")
if len(overlap) < 10:
    print(" Overlap is LOW. Fix sample_id in metadata to match TPM column names exactly.")

tpm  = tpm.loc[:, overlap]
meta = meta.set_index("sample_id").loc[overlap].reset_index()

print("Final TPM:", tpm.shape)
print("Final META:", meta.shape)

meta.to_csv(f"{OUT_PROC}/ibd_metadata_clean.csv", index=False)
print("Saved ->", f"{OUT_PROC}/ibd_metadata_clean.csv")

### log2 transform


In [ ]:
expr_log = np.log2(tpm + 1)
expr_log.to_csv(f"{OUT_PROC}/ibd_log2_tpm.csv")
print("Saved ->", f"{OUT_PROC}/ibd_log2_tpm.csv")

### DEG function


In [ ]:
def run_deg(expr_log_df, meta_df, g1, g2, group_col="group"):
    groups = meta_df[group_col].astype(str).values
    i1 = np.where(groups == g1)[0]
    i2 = np.where(groups == g2)[0]
    print(f"Group sizes: {g1}={len(i1)}, {g2}={len(i2)}")

    X1 = expr_log_df.iloc[:, i1]
    X2 = expr_log_df.iloc[:, i2]

    mean1, mean2 = X1.mean(1), X2.mean(1)
    logFC = mean1 - mean2

    pvals = np.array([
        ttest_ind(X1.iloc[i,:], X2.iloc[i,:], equal_var=False).pvalue
        for i in range(expr_log_df.shape[0])
    ])
    padj = multipletests(pvals, method="fdr_bh")[1]

    out = pd.DataFrame({
        "gene": expr_log_df.index.astype(str),
        f"mean_{g1}": mean1.values,
        f"mean_{g2}": mean2.values,
        f"logFC_{g1}_minus_{g2}": logFC.values,
        "pval": pvals,
        "padj_fdr": padj
    }).sort_values("padj_fdr")
    return out



### Run UC vs Control and CD vs Control


In [ ]:
LABEL_CONTROL="Control"
LABEL_UC="UC"
LABEL_CD="CD"

print("Group counts:")
display(meta["group"].value_counts())

deg_uc = run_deg(expr_log, meta, LABEL_UC, LABEL_CONTROL)
deg_cd = run_deg(expr_log, meta, LABEL_CD, LABEL_CONTROL)

deg_uc.to_csv(f"{OUT_DE}/IBD_DEG_{LABEL_UC}_vs_{LABEL_CONTROL}.csv", index=False)
deg_cd.to_csv(f"{OUT_DE}/IBD_DEG_{LABEL_CD}_vs_{LABEL_CONTROL}.csv", index=False)

print("Saved UC ->", f"{OUT_DE}/IBD_DEG_{LABEL_UC}_vs_{LABEL_CONTROL}.csv")
print("Saved CD ->", f"{OUT_DE}/IBD_DEG_{LABEL_CD}_vs_{LABEL_CONTROL}.csv")
display(deg_uc.head(10))


### Volcano (UC vs Control)

In [ ]:
df = deg_uc.copy()
df["neglog10_fdr"] = -np.log10(df["padj_fdr"] + 1e-300)

plt.figure(figsize=(6,4))
plt.scatter(df[f"logFC_{LABEL_UC}_minus_{LABEL_CONTROL}"], df["neglog10_fdr"], alpha=0.35)
plt.axvline(1, linestyle="--"); plt.axvline(-1, linestyle="--")
plt.axhline(-np.log10(0.05), linestyle="--")
plt.title(f"IBD Volcano: {LABEL_UC} vs {LABEL_CONTROL}")
plt.xlabel("log2FC"); plt.ylabel("-log10(FDR)")
plt.tight_layout()
plt.savefig(f"{OUT_FIG}/volcano_{LABEL_UC}_vs_{LABEL_CONTROL}.png", dpi=200)
plt.show()